In [20]:
import pandas as pd
from pathlib import Path

candidates = pd.read_csv("../raw/lok_sabha_candidate_dfs/candidates_all_years.csv")
links = pd.read_csv("../raw/lok_sabha_constituencies/constituency_links_all_years.csv")

print("Candidates:", candidates.shape)
print("Links:", links.shape)

display(candidates.head())
display(links.head())

print("Candidate columns:", candidates.columns.tolist())
print("Link columns:", links.columns.tolist())

Candidates: (10395, 23)
Links: (1679, 10)


,year,state,state_id,const,const_id,sno,candidate,candidate_id,candidate_url,is_winner,...,age,total_assets,asset_link,liabilities,liability_link,constituency_url,html_path,is_byelection,bye_election_date,election_date
0,2024,BIHAR,5,ARARIA,56,6,Pradeep Kumar Singh,3627,https://www.myneta.info/LokSabha2024/candidate...,True,...,58,NaN,https://myneta.info/image_v2.php?myneta_folder...,NaN,https://myneta.info/image_v2.php?myneta_folder...,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024
1,2024,BIHAR,5,ARARIA,56,1,Akhilesh Kumar,4040,https://www.myneta.info/LokSabha2024/candidate...,False,...,47,"Rs 2,47,11,288 ~ 2 Crore+",NaN,"Rs 3,05,374 ~ 3 Lacs+",NaN,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024
2,2024,BIHAR,5,ARARIA,56,2,Md Ghousul Azam,4773,https://www.myneta.info/LokSabha2024/candidate...,False,...,38,"Rs 1,60,50,000 ~ 1 Crore+",NaN,"Rs 14,00,000 ~ 14 Lacs+",NaN,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024
3,2024,BIHAR,5,ARARIA,56,3,Md. Ismail,4107,https://www.myneta.info/LokSabha2024/candidate...,False,...,56,NaN,https://myneta.info/image_v2.php?myneta_folder...,NaN,https://myneta.info/image_v2.php?myneta_folder...,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024
4,2024,BIHAR,5,ARARIA,56,4,Md. Mobinul Haque,4772,https://www.myneta.info/LokSabha2024/candidate...,False,...,51,"Rs 34,05,000 ~ 34 Lacs+",NaN,Rs 0 ~,NaN,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html,False,NaN,07-05-2024


,year,state_ut,state_id,const,const_id,link,anchor_title,election_date,is_byelection,bye_election_date
0,2024,ANDAMAN AND NICOBAR ISLANDS,1,ANDAMAN AND NICOBAR ISLANDS,579,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 19-04-2024,19-04-2024,False,NaN
1,2024,ANDHRA PRADESH,2,AMALAPURAM (SC),7,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 13-05-2024,13-05-2024,False,NaN
2,2024,ANDHRA PRADESH,2,ANAKAPALLE,5,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 13-05-2024,13-05-2024,False,NaN
3,2024,ANDHRA PRADESH,2,ANANTHAPUR,20,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 13-05-2024,13-05-2024,False,NaN
4,2024,ANDHRA PRADESH,2,ARAKU (ST),1,https://www.myneta.info/LokSabha2024/index.php...,Date of Election 13-05-2024,13-05-2024,False,NaN


Candidate columns: ['year', 'state', 'state_id', 'const', 'const_id', 'sno', 'candidate', 'candidate_id', 'candidate_url', 'is_winner', 'party', 'criminal_cases', 'education', 'age', 'total_assets', 'asset_link', 'liabilities', 'liability_link', 'constituency_url', 'html_path', 'is_byelection', 'bye_election_date', 'election_date']
Link columns: ['year', 'state_ut', 'state_id', 'const', 'const_id', 'link', 'anchor_title', 'election_date', 'is_byelection', 'bye_election_date']


In [21]:
def to_bool(s):
    return s.astype("string").str.strip().str.casefold().map({
        "true": True, "1": True, "yes": True, "y": True,
        "false": False, "0": False, "no": False, "n": False
    }).fillna(False).astype(bool)

def url_key(s):
    return s.astype("string").str.strip().str.rstrip("/")

candidates["is_winner"] = to_bool(candidates["is_winner"])
candidates["is_byelection"] = to_bool(candidates["is_byelection"])
links["is_byelection"] = to_bool(links["is_byelection"])

candidates["event_key"] = url_key(candidates["constituency_url"])
links["event_key"] = url_key(links["link"])

print("Missing event URLs:", candidates["event_key"].isna().sum())
print("Unique candidate events:", candidates["event_key"].nunique())
print("By-election candidate rows:", candidates["is_byelection"].sum())

Missing event URLs: 0
Unique candidate events: 637
By-election candidate rows: 167


In [22]:
candidates["candidate_key"] = candidates["candidate_id"].astype("string").str.strip()

missing_id = candidates["candidate_key"].isna() | candidates["candidate_key"].eq("")
candidates.loc[missing_id, "candidate_key"] = (
    "name:" + candidates.loc[missing_id, "candidate"]
    .astype("string").str.casefold().str.strip()
)

duplicates = candidates.duplicated(["event_key", "candidate_key"], keep=False)

print("Duplicate candidate-event rows:", int(duplicates.sum()))
display(candidates.loc[duplicates].sort_values(["event_key", "candidate"]))

Duplicate candidate-event rows: 0


,year,state,state_id,const,const_id,sno,candidate,candidate_id,candidate_url,is_winner,...,asset_link,liabilities,liability_link,constituency_url,html_path,is_byelection,bye_election_date,election_date,event_key,candidate_key


In [23]:
winner_counts = candidates.groupby("event_key")["is_winner"].sum()

print("Events with zero winners:", int((winner_counts == 0).sum()))
print("Events with multiple winners:", int((winner_counts > 1).sum()))

display(winner_counts[winner_counts != 1].rename("winner_count").to_frame())

Events with zero winners: 3
Events with multiple winners: 0


,winner_count
event_key,
https://www.myneta.info/LokSabha2019/index.php?action=show_candidates&constituency_id=905,0
https://www.myneta.info/ls2014/index.php?action=show_candidates&constituency_id=336,0
https://www.myneta.info/ls2014/index.php?action=show_candidates&constituency_id=431,0


In [24]:
candidate_urls = set(candidates["event_key"].dropna())
link_urls = set(links["event_key"].dropna())

print("Candidate URLs absent from links:", len(candidate_urls - link_urls))
print("Link URLs absent from candidates:", len(link_urls - candidate_urls))

Candidate URLs absent from links: 0
Link URLs absent from candidates: 1042


In [25]:
winners = candidates[candidates["is_winner"]].copy()

multi_wins = (
    winners.groupby(["candidate_key", "year"])
    .agg(
        candidate=("candidate", "first"),
        wins=("event_key", "nunique"),
        constituencies=("const", lambda s: list(pd.unique(s))),
        byelection_events=("is_byelection", "sum"),
    )
    .query("wins > 1")
)

display(multi_wins)

,,candidate,wins,constituencies,byelection_events
candidate_key,year,,,,


In [26]:
general = candidates[~candidates["is_byelection"]].copy()
general["party_clean"] = general["party"].astype("string").str.casefold().str.strip()

party_check = (
    general.dropna(subset=["candidate_key", "party_clean"])
    .groupby(["candidate_key", "year"])
    .agg(
        candidate=("candidate", "first"),
        constituencies=("const", "nunique"),
        parties=("party_clean", lambda s: sorted(set(s))),
    )
)

display(party_check[
    (party_check["constituencies"] > 1) &
    (party_check["parties"].map(len) > 1)
])

,,candidate,constituencies,parties
candidate_key,year,,,


In [27]:
seat_year_wins = (
    winners.groupby(["state", "const", "year"])
    .agg(
        winner_count=("candidate_key", "nunique"),
        winners=("candidate", lambda s: list(pd.unique(s))),
        has_byelection=("is_byelection", "any"),
    )
)

display(seat_year_wins[seat_year_wins["winner_count"] > 1])

winner_count  \
state         const         year                 
BIHAR         ARARIA        2014             2   
              VALMIKI NAGAR 2019             2   
MAHARASHTRA   BEED          2014             2   
              NANDED        2024             2   
              PALGHAR       2014             2   
              SATARA        2019             2   
TAMIL NADU    KANNIYAKUMARI 2019             2   
UTTAR PRADESH AZAMGARH      2019             2   
              GORAKHPUR     2014             2   
              KAIRANA       2014             2   
              MAINPURI      2014             2   
                            2019             2   
              PHULPUR       2014             2   
              RAMPUR        2019             2   

                                                                            winners  \
state         const         year                                                      
BIHAR         ARARIA        2014                      [Taslim Uddin, Sarfaraz Alam]   
              VALMIKI NAGAR 2019             [Baidyanath Prasad Mahto, Sunil Kumar]   
MAHARASHTRA   BEED          2014        [Munde Gopinathrao Pandurang, Munde Pritam]   
              NANDED        2024  [Vasantrao Balwantrao Chavan, Chavan Ravindra ...   
              PALGHAR       2014  [Wanaga Chintaman Navasha, Gavit Rajendra Dhedya]   
              SATARA        2019  [Bhonsle Shrimant Chhatrapati Udayanraje Prata...   
TAMIL NADU    KANNIYAKUMARI 2019  [Vasanthakumar H, Vijayakumar Alias Viyay Vasa...   
UTTAR PRADESH AZAMGARH      2019       [Akhilesh Yadav, Dinesh Lal Yadav "Nirahua"]   
              GORAKHPUR     2014                  [Adityanath, Pravin Kumar Nishad]   
              KAIRANA       2014                      [Hukum Singh, Tabassum Begum]   
              MAINPURI      2014      [Mulayam Singh Yadav, Tej Pratap Singh Yadav]   
                            2019                [Mulayam Singh Yadav, Dimple Yadav]   
              PHULPUR       2014       [Keshav Prasad, Nagendra Pratap Singh Patel]   
              RAMPUR        2019        [Mohammad Azam Khan, Ghanshyam Singh Lodhi]   

                                  has_byelection  
state         const         year                  
BIHAR         ARARIA        2014            True  
              VALMIKI NAGAR 2019            True  
MAHARASHTRA   BEED          2014            True  
              NANDED        2024            True  
              PALGHAR       2014            True  
              SATARA        2019            True  
TAMIL NADU    KANNIYAKUMARI 2019            True  
UTTAR PRADESH AZAMGARH      2019            True  
              GORAKHPUR     2014            True  
              KAIRANA       2014            True  
              MAINPURI      2014            True  
                            2019            True  
              PHULPUR       2014            True  
              RAMPUR        2019            True

In [28]:
candidates["election_date_parsed"] = pd.to_datetime(
    candidates["election_date"], dayfirst=True, errors="coerce"
)
candidates["bye_date_parsed"] = pd.to_datetime(
    candidates["bye_election_date"], dayfirst=True, errors="coerce"
)

regular = ~candidates["is_byelection"]

print("Regular rows missing election date:",
      candidates.loc[regular, "election_date_parsed"].isna().sum())
print("By-election rows missing by-election date:",
      candidates.loc[~regular, "bye_date_parsed"].isna().sum())
print("Regular rows with a by-election date:",
      candidates.loc[regular, "bye_date_parsed"].notna().sum())

display(candidates.loc[
    (regular & candidates["election_date_parsed"].isna()) |
    (~regular & candidates["bye_date_parsed"].isna()),
    ["year", "state", "const", "candidate", "is_byelection",
     "election_date", "bye_election_date"]
].drop_duplicates(["year", "state", "const"]))

Regular rows missing election date: 0
By-election rows missing by-election date: 0
Regular rows with a by-election date: 0


,year,state,const,candidate,is_byelection,election_date,bye_election_date


In [29]:
events = links.copy()

events["election_dt"] = pd.to_datetime(
    events["election_date"], dayfirst=True, errors="coerce"
)
events["bye_dt"] = pd.to_datetime(
    events["bye_election_date"], dayfirst=True, errors="coerce"
)

events["_state"] = (
    events["state_ut"].astype("string").str.casefold().str.strip()
)
events["_const"] = (
    events["const"].astype("string").str.casefold().str.strip()
)

# One record per event URL; do not merge candidate rows.
events = events.drop_duplicates("event_key")

regular_events = events[
    ~events["is_byelection"] & events["election_dt"].notna()
]

general_dates = {
    key: sorted(group["election_dt"].dropna().unique())
    for key, group in regular_events.groupby(["_state", "_const"])
}

results = []

for _, row in events[events["is_byelection"]].iterrows():
    date = row["bye_dt"]
    key = (row["_state"], row["_const"])
    dates = general_dates.get(key, [])

    previous = [d for d in dates if pd.notna(date) and d < date]
    following = [d for d in dates if pd.notna(date) and d > date]

    prev_date = max(previous) if previous else pd.NaT
    next_date = min(following) if following else pd.NaT

    if pd.isna(date):
        status = "MISSING BY-ELECTION DATE"
    elif pd.isna(prev_date):
        status = "NO PREVIOUS GENERAL ELECTION"
    elif pd.isna(next_date):
        status = "NO NEXT GENERAL ELECTION"
    elif prev_date < date < next_date:
        status = "PASS"
    else:
        status = "FAIL"

    results.append({
        "year": row["year"],
        "state": row["state_ut"],
        "const": row["const"],
        "bye_election_date": date,
        "previous_general_election": prev_date,
        "next_general_election": next_date,
        "status": status,
        "link": row["link"],
    })

bye_date_check = pd.DataFrame(results)

if not bye_date_check.empty:
    print(bye_date_check["status"].value_counts())
    display(bye_date_check[bye_date_check["status"] != "PASS"])
else:
    print("No by-election events found.")

status
PASS                            39
NO NEXT GENERAL ELECTION         7
NO PREVIOUS GENERAL ELECTION     4
Name: count, dtype: int64


,year,state,const,bye_election_date,previous_general_election,next_general_election,status,link
0,2024,ASSAM,NAGAON,2026-10-06,2024-04-26,NaT,NO NEXT GENERAL ELECTION,https://www.myneta.info/LokSabha2024/index.php...
1,2024,KERALA,WAYANAD,2024-11-13,2024-04-26,NaT,NO NEXT GENERAL ELECTION,https://www.myneta.info/LokSabha2024/index.php...
2,2024,MAHARASHTRA,NANDED,2024-11-20,2024-04-26,NaT,NO NEXT GENERAL ELECTION,https://www.myneta.info/LokSabha2024/index.php...
3,2019,ANDHRA PRADESH,TIRUPATI,2021-04-17,2019-04-11,NaT,NO NEXT GENERAL ELECTION,https://www.myneta.info/LokSabha2019/index.php...
4,2019,BIHAR,SAMASTIPUR (SC),2019-10-21,NaT,2024-05-13,NO PREVIOUS GENERAL ELECTION,https://www.myneta.info/LokSabha2019/index.php...
6,2019,DADRA & NAGAR HAVELI,DADRA AND NAGAR HAVELI,2021-10-30,2019-04-23,NaT,NO NEXT GENERAL ELECTION,https://www.myneta.info/LokSabha2019/index.php...
12,2019,PUNJAB,JALANDHAR,2023-05-10,2019-05-19,NaT,NO NEXT GENERAL ELECTION,https://www.myneta.info/LokSabha2019/index.php...
29,2014,MADHYA PRADESH,SHAHDOL (ST),2016-11-19,NaT,2024-04-19,NO PREVIOUS GENERAL ELECTION,https://www.myneta.info/ls2014/index.php?actio...
31,2014,MAHARASHTRA,BHANDARA-GONDIYA,2018-05-28,NaT,NaT,NO PREVIOUS GENERAL ELECTION,https://www.myneta.info/ls2014/index.php?actio...
35,2014,ORISSA,KANDHAMAL,2014-10-15,2014-04-10,NaT,NO NEXT GENERAL ELECTION,https://www.myneta.info/ls2014/index.php?actio...


In [30]:
candidates["effective_date"] = candidates["election_date_parsed"]

mask = candidates["is_byelection"]
candidates.loc[mask, "effective_date"] = (
    candidates.loc[mask, "bye_date_parsed"]
    .combine_first(candidates.loc[mask, "election_date_parsed"])
)

dated = candidates.dropna(subset=["effective_date"]).copy()

latest_date = dated.groupby(["state", "const"])["effective_date"].transform("max")
latest_rows = dated[dated["effective_date"].eq(latest_date)]
final_winners = latest_rows[latest_rows["is_winner"]]

display(final_winners[
    ["state", "const", "year", "effective_date",
     "is_byelection", "candidate", "party"]
].sort_values(["state", "const"]))

print("Seats with multiple latest winners:",
      final_winners.groupby(["state", "const"])["candidate_key"]
      .nunique().gt(1).sum())

,state,const,year,effective_date,is_byelection,candidate,party
0,BIHAR,ARARIA,2024,2024-05-07,False,Pradeep Kumar Singh,BJP
9,BIHAR,ARRAH,2024,2024-06-01,False,Sudama Prasad,CPI(ML)(L)
23,BIHAR,AURANGABAD,2024,2024-04-19,False,Abhay Kumar Sinha,RJD
32,BIHAR,BANKA,2024,2024-04-26,False,Giridhari Yadav,JD(U)
42,BIHAR,BEGUSARAI,2024,2024-05-13,False,Giriraj Singh,BJP
...,...,...,...,...,...,...,...
3359,UTTAR PRADESH,SHRAWASTI,2024,2024-05-25,False,Ram Shiromani Verma,SP
3371,UTTAR PRADESH,SITAPUR,2024,2024-05-13,False,Rakesh Rathor,INC
3379,UTTAR PRADESH,SULTANPUR,2024,2024-05-25,False,Rambhual Nishad,SP
3388,UTTAR PRADESH,UNNAO,2024,2024-05-13,False,Swami Sachchidanand Hari Sakshi,BJP


Seats with multiple latest winners: 0


In [31]:
print("Candidate rows:", len(candidates))
print("Unique events:", candidates["event_key"].nunique())
print("Winner rows:", int(candidates["is_winner"].sum()))
print("Events without exactly one winner:", int((winner_counts != 1).sum()))
print("Duplicate candidate-event rows:", int(duplicates.sum()))
print("Missing event URLs:", int(candidates["event_key"].isna().sum()))
print("Missing effective dates:", int(candidates["effective_date"].isna().sum()))

if not bye_date_check.empty:
    print("\nBy-election date validation:")
    print(bye_date_check["status"].value_counts())

Candidate rows: 10395
Unique events: 637
Winner rows: 634
Events without exactly one winner: 3
Duplicate candidate-event rows: 0
Missing event URLs: 0
Missing effective dates: 0

By-election date validation:
status
PASS                            39
NO NEXT GENERAL ELECTION         7
NO PREVIOUS GENERAL ELECTION     4
Name: count, dtype: int64


In [33]:
display(
    winner_counts[winner_counts != 1]
    .rename("winner_count")
    .to_frame()
    .join(
        candidates.groupby("event_key").agg(
            year=("year", "first"),
            state=("state", "first"),
            constituency=("const", "first"),
            is_byelection=("is_byelection", "first"),
            candidates=("candidate", list),
        )
    )
)

,winner_count,year,state,constituency,is_byelection,candidates
event_key,,,,,,
https://www.myneta.info/LokSabha2019/index.php?action=show_candidates&constituency_id=905,0,2019,UTTAR PRADESH,AKBARPUR,False,"[Amit Tripathi, Arun Kumar, Ashok Paswan, Mahe..."
https://www.myneta.info/ls2014/index.php?action=show_candidates&constituency_id=336,0,2014,MAHARASHTRA,MUMBAI NORTH WEST,False,"[Bhole Pushpa Milind, Kamat Gurudas Vasant, Ma..."
https://www.myneta.info/ls2014/index.php?action=show_candidates&constituency_id=431,0,2014,UTTAR PRADESH,FARRUKHABAD,False,"[Anju, Anokhey, Anshu Mahan, Anurag Rajput, Da..."


In [36]:
events = links.copy()

events["election_dt"] = pd.to_datetime(
    events["election_date"], dayfirst=True, errors="coerce"
)
events["bye_dt"] = pd.to_datetime(
    events["bye_election_date"], dayfirst=True, errors="coerce"
)

events["_state"] = (
    events["state_ut"].astype("string").str.casefold().str.strip()
)
events["_const"] = (
    events["const"].astype("string").str.casefold().str.strip()
)
events["is_byelection"] = to_bool(events["is_byelection"])

events = events.drop_duplicates("event_key")

regular = events[
    ~events["is_byelection"] & events["election_dt"].notna()
]

general_dates = {
    key: sorted(group["election_dt"].dropna().unique())
    for key, group in regular.groupby(["_state", "_const"])
}

results = []

for _, row in events[events["is_byelection"]].iterrows():
    date = row["bye_dt"]
    key = (row["_state"], row["_const"])
    dates = general_dates.get(key, [])

    previous = [d for d in dates if pd.notna(date) and d < date]
    following = [d for d in dates if pd.notna(date) and d > date]

    prev_date = max(previous) if previous else pd.NaT
    next_date = min(following) if following else pd.NaT

    if pd.isna(date):
        status = "MISSING BY-ELECTION DATE"
    elif pd.isna(prev_date):
        status = "NO PREVIOUS GENERAL ELECTION"
    elif pd.notna(next_date) and not (prev_date < date < next_date):
        status = "BY-ELECTION OUTSIDE GENERAL-ELECTION INTERVAL"
    elif pd.isna(next_date) and int(row["year"]) != 2024:
        status = "NO NEXT GENERAL ELECTION IN DATA"
    else:
        status = "PASS"

    results.append({
        "year": row["year"],
        "state": row["state_ut"],
        "const": row["const"],
        "bye_election_date": date,
        "previous_general_election": prev_date,
        "next_general_election": next_date,
        "status": status,
        "link": row["link"],
    })

bye_date_check = pd.DataFrame(results)

print(bye_date_check["status"].value_counts())
display(bye_date_check[bye_date_check["status"] != "PASS"])

status
PASS                                42
NO NEXT GENERAL ELECTION IN DATA     4
NO PREVIOUS GENERAL ELECTION         4
Name: count, dtype: int64


,year,state,const,bye_election_date,previous_general_election,next_general_election,status,link
3,2019,ANDHRA PRADESH,TIRUPATI,2021-04-17,2019-04-11,NaT,NO NEXT GENERAL ELECTION IN DATA,https://www.myneta.info/LokSabha2019/index.php...
4,2019,BIHAR,SAMASTIPUR (SC),2019-10-21,NaT,2024-05-13,NO PREVIOUS GENERAL ELECTION,https://www.myneta.info/LokSabha2019/index.php...
6,2019,DADRA & NAGAR HAVELI,DADRA AND NAGAR HAVELI,2021-10-30,2019-04-23,NaT,NO NEXT GENERAL ELECTION IN DATA,https://www.myneta.info/LokSabha2019/index.php...
12,2019,PUNJAB,JALANDHAR,2023-05-10,2019-05-19,NaT,NO NEXT GENERAL ELECTION IN DATA,https://www.myneta.info/LokSabha2019/index.php...
29,2014,MADHYA PRADESH,SHAHDOL (ST),2016-11-19,NaT,2024-04-19,NO PREVIOUS GENERAL ELECTION,https://www.myneta.info/ls2014/index.php?actio...
31,2014,MAHARASHTRA,BHANDARA-GONDIYA,2018-05-28,NaT,NaT,NO PREVIOUS GENERAL ELECTION,https://www.myneta.info/ls2014/index.php?actio...
35,2014,ORISSA,KANDHAMAL,2014-10-15,2014-04-10,NaT,NO NEXT GENERAL ELECTION IN DATA,https://www.myneta.info/ls2014/index.php?actio...
47,2014,WEST BENGAL,COOCHBEHAR (SC),2016-11-19,NaT,NaT,NO PREVIOUS GENERAL ELECTION,https://www.myneta.info/ls2014/index.php?actio...


In [38]:
# Count unique constituencies by year and state, separating general and by-elections

df = candidates.copy()
df["is_byelection"] = to_bool(df["is_byelection"])

counts = (
    df.groupby(["year", "state", "is_byelection"])["const_id"]
    .nunique()
    .unstack(fill_value=0)
    .rename(columns={
        False: "general_election_constituencies",
        True: "by_election_constituencies"
    })
    .reset_index()
)

for col in ["general_election_constituencies", "by_election_constituencies"]:
    if col not in counts.columns:
        counts[col] = 0

counts["year"] = pd.to_numeric(counts["year"])
counts = counts.sort_values(["year", "state"]).reset_index(drop=True)

display(counts)

is_byelection,year,state,general_election_constituencies,by_election_constituencies
0,2014,BIHAR,40,1
1,2014,MAHARASHTRA,48,3
2,2014,TAMIL NADU,39,0
3,2014,UTTAR PRADESH,80,4
4,2019,BIHAR,40,2
5,2019,MAHARASHTRA,48,1
6,2019,TAMIL NADU,39,1
7,2019,UTTAR PRADESH,80,3
8,2024,BIHAR,40,0
9,2024,MAHARASHTRA,48,1
